# Exercițiu Laborator 1 — Simularea variabilelor aleatoare

## Enunțul problemei
Considerăm o urnă în care avem **3 bile roșii**, **4 bile albastre** și **2 bile negre** (în total 9 bile). Aruncăm un zar corect cu 6 fețe:
* dacă obținem un **număr prim** ($\{2, 3, 5\}$), adăugăm o **bilă neagră** în urnă;
* dacă obținem **6** ($\{6\}$), adăugăm o **bilă roșie** în urnă;
* în **celelalte cazuri** ($\{1, 4\}$), adăugăm o **bilă albastră** în urnă.

Apoi extragem o bilă din urnă.

**Cerințe:**
* **a)** Simulați în Python experimentul de mai sus.
* **b)** Folosind modelarea făcută, estimați probabilitatea de a obține o bilă roșie.
* **c)** Calculați probabilitatea teoretică a evenimentului de mai sus și comparați-o cu cea obținută prin simulare.

## a) & b) Simularea experimentului în Python și estimarea probabilității

Modelăm un singur experiment într-o funcție `simuleaza_experiment()` care:
1. Inițializează urna cu `['R']*3 + ['A']*4 + ['N']*2`.
2. Generează aruncarea zarului $Z \in \{1, 2, 3, 4, 5, 6\}$.
3. Actualizează conținutul urnei în funcție de rezultatul zarului:
   * $Z \in \{2, 3, 5\} \implies$ adaugă `'N'`;
   * $Z = 6 \implies$ adaugă `'R'`;
   * $Z \in \{1, 4\} \implies$ adaugă `'A'`.
4. Extrage uniform aleator o bilă din cele 10 bile aflate acum în urnă.

Repetăm experimentul de $N = 200\,000$ ori (metoda Monte Carlo) pentru a estima probabilitatea $P(\text{Bilă Roșie})$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Fixăm seed-ul pentru reproductibilitate
rng = np.random.default_rng(42)

def simuleaza_experiment(rng):
    # Urna inițială: 3 roșii, 4 albastre, 2 negre
    urna = ['R'] * 3 + ['A'] * 4 + ['N'] * 2
    
    # Aruncarea unui zar corect (1..6)
    zar = rng.integers(1, 7)
    
    if zar in (2, 3, 5):      # număr prim
        urna.append('N')
    elif zar == 6:            # fața 6
        urna.append('R')
    else:                     # celelalte cazuri: {1, 4}
        urna.append('A')
        
    # Extragem aleator o bilă din urnă
    bila_extrasa = rng.choice(urna)
    return bila_extrasa

# Exemplu pentru un singur experiment
print("Rezultatul unui singur experiment:", simuleaza_experiment(rng))

# Simularea Monte Carlo pe N = 200.000 de repetări
N = 200_000
rezultate = np.array([simuleaza_experiment(rng) for _ in range(N)])

prob_estimata = np.mean(rezultate == 'R')
prob_teoretica = 19 / 60

print(f"Număr de simulări N = {N:,}")
print(f"Probabilitatea estimată (Monte Carlo): {prob_estimata:.6f}")
print(f"Probabilitatea teoretică (19/60):      {prob_teoretica:.6f}")
print(f"Eroarea absolută:                      {abs(prob_estimata - prob_teoretica):.6f}")

## c) Calculul probabilității teoretice și comparația cu simularea

Aplicăm **Formula probabilității totale**. Notăm cu $R$ evenimentul de a extrage o bilă roșie și considerăm partiția spațiului de probă dată de rezultatul aruncării zarului:

1. **$E_1$: zarul indică un număr prim ($\{2, 3, 5\}$)**
   * $P(E_1) = \frac{3}{6} = \frac{1}{2}$.
   * Se adaugă o bilă neagră $\implies$ urna conține **3 roșii, 4 albastre, 3 negre** (10 bile).
   * $P(R \mid E_1) = \frac{3}{10}$.

2. **$E_2$: zarul indică 6 ($\{6\}$)**
   * $P(E_2) = \frac{1}{6}$.
   * Se adaugă o bilă roșie $\implies$ urna conține **4 roșii, 4 albastre, 2 negre** (10 bile).
   * $P(R \mid E_2) = \frac{4}{10}$.

3. **$E_3$: zarul indică alt număr ($\{1, 4\}$)**
   * $P(E_3) = \frac{2}{6} = \frac{1}{3}$.
   * Se adaugă o bilă albastră $\implies$ urna conține **3 roșii, 5 albastre, 2 negre** (10 bile).
   * $P(R \mid E_3) = \frac{3}{10}$.

Prin formula probabilității totale:
$$
P(R) = P(E_1)P(R \mid E_1) + P(E_2)P(R \mid E_2) + P(E_3)P(R \mid E_3)
$$
$$
P(R) = \frac{3}{6} \cdot \frac{3}{10} + \frac{1}{6} \cdot \frac{4}{10} + \frac{2}{6} \cdot \frac{3}{10} = \frac{9 + 4 + 6}{60} = \frac{19}{60} \approx 0.316667.
$$

**Comparație:** Estimarea obținută prin simulare Monte Carlo ($\approx 0.3167$) converge către valoarea teoretică exactă $\frac{19}{60} \approx 0.316667$, diferența fiind de ordinul $10^{-4}$ (conform Legii Numerelor Mari).

In [ ]:
# Vizualizarea convergenței estimării Monte Carlo către probabilitatea teoretică
estimari_cumulative = np.cumsum(rezultate == 'R') / np.arange(1, N + 1)

plt.figure(figsize=(10, 4.5))
plt.plot(np.arange(1, N + 1), estimari_cumulative, label="Estimare Monte Carlo", color="#1f77b4", linewidth=1.2)
plt.axhline(prob_teoretica, color="#d62728", linestyle="--", linewidth=1.5, label=f"Probabilitate teoretică (19/60 ≈ {prob_teoretica:.4f})")
plt.xscale("log")
plt.xlabel("Număr de simulări (scară logaritmică)")
plt.ylabel("Probabilitate estimată P(Roșie)")
plt.title("Convergența estimării Monte Carlo către probabilitatea teoretică")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()